In [1]:
import librosa
import IPython.display as ipd
import numpy as np
from scipy.signal import lfilter
import soundfile as sf

In [2]:
filename = '../../data/tomsdiner_mono.wav'
diner, srate = librosa.load(filename)
ipd.Audio(diner, rate=srate)

## REVERB/SPATIALIZATION ALGORITHMS

Messing around with my own code (pingpong) then working on a python implementation of freeverb (https://ccrma.stanford.edu/~jos/pasp/Freeverb.html). I'm familiar with the implementation in passing but I unfortunately had to get some AI assistance with the implementation, which I'm ashamed of.

In [3]:
# Bounce audio left and right
def ping_pong(x, delay, sr, decay=0.7, h=(0.5, 0.5), tail=3.0):
    D = int(round(delay))
    out = np.zeros((len(x) + int(tail * sr), 2))
    out[:len(x)] += x[:, None]
    tap = x.copy()
    peak = np.max(np.abs(tap)) + 1e-12
    r = 0
    while True:
        r += 1
        tap = decay * lfilter(h, [1.0], tap)
        if np.max(np.abs(tap)) < 1e-4 * peak or r * D >= len(out):
            break
        end = min(len(out), r * D + len(tap))
        out[r * D:end, r % 2] += tap[:end - r * D] # [:, r%2] grabs left and right channels alternating
    return np.transpose(out) # [[1, 2], [3, 4], [5, 6]] -> [[1, 3, 5], [2, 4, 6]]

# helpers
def _rms(a):
    return np.sqrt(np.mean(np.asarray(a) ** 2)) + 1e-12

def _damped_comb(x, delay, rt60, damp, sr):
    fb = 10.0 ** (-3.0 * delay / (sr * rt60))
    a = np.zeros(delay + 2)
    a[0], a[1] = 1.0, -damp
    a[delay] -= fb * (1 - damp)
    b = np.zeros(delay + 2)
    b[delay], b[delay + 1] = 1.0, -damp
    return lfilter(b, a, x)
 
 
def _allpass(x, delay, g=0.5):
    b = np.zeros(delay + 1)
    a = np.zeros(delay + 1)
    b[0], b[delay] = -g, 1.0
    a[0], a[delay] = 1.0, -g
    return lfilter(b, a, x)

def _finish(x, wet, sr, mix, predelay):
    n_out = len(wet)
    if predelay > 0:
        pd = int(predelay * sr)
        wet = np.vstack([np.zeros((pd, 2)), wet])[:n_out]
    wet = wet * (_rms(x) / _rms(wet[:len(x)]))
    dry = np.zeros(n_out)
    dry[:len(x)] = x
    m = np.asarray(mix, float)
    if m.ndim:
        m = np.interp(np.linspace(0, 1, n_out), np.linspace(0, 1, len(m)), m)
        m = m[:, None]
    return (1 - m) * dry[:, None] + m * wet

# Freeverb implementation (https://ccrma.stanford.edu/~jos/pasp/Freeverb.html)
def freeverb(x, sr, decay=2.0, damp=0.3, mix=0.3, predelay=0.0, tail=None, spread=23):
    s = sr / 44100
    tail_n = int((decay if tail is None else tail) * sr)
    xp = np.concatenate([x, np.zeros(tail_n)])
    wet = np.zeros((len(xp), 2))
    comb = [1116, 1188, 1277, 1356, 1422, 1491, 1557, 1617]
    ap = [556, 441, 341, 225]
    for ch in range(2):
        off = ch * int(spread * s)
        acc = sum(_damped_comb(xp, int(d * s) + off, decay, damp, sr) for d in comb)
        for d in ap:
            acc = _allpass(acc, int(d * s) + off)
        wet[:, ch] = acc
    return np.transpose(_finish(x, wet, sr, mix, predelay))


pingpong = ping_pong(diner, 5000, srate)
pad = freeverb(diner, srate, decay=np.inf, damp=0.5, mix=1.0, tail=10)
swell = freeverb(diner[::-1], srate, decay=3, mix=0.7, tail=3)[::-1]

sf.write("pingpong.wav", np.transpose(pingpong / np.max(np.abs(pingpong))), srate)
sf.write("pad.wav", np.transpose(pad / np.max(np.abs(pad))), srate)
sf.write("swell.wav", np.transpose(swell / np.max(np.abs(swell))), srate)

ipd.display(ipd.Audio(pingpong, rate=srate))
ipd.display(ipd.Audio(pad, rate=srate))
ipd.display(ipd.Audio(swell, rate=srate))